## Save Your Work

Before you begin, save a copy of this notebook to your Google Drive: **File → Save a copy in Drive**.

# Module 13 Assessment — Understanding ML Models

Compare five advanced algorithms against logistic regression and decision tree baselines on the breast cancer dataset.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import roc_auc_score, classification_report, RocCurveDisplay, recall_score
from sklearn.pipeline import Pipeline

cancer = load_breast_cancer()
X = pd.DataFrame(cancer.data, columns=cancer.feature_names)
y = pd.Series(cancer.target)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Positive (malignant) class in train: {(y_train==0).sum()}")
# Train: (455, 30), Test: (114, 30)
# Positive (malignant) class in train: 170

## Task 1: Train All Models

For each model below, fit on the scaled training data and compute test accuracy, recall (malignant class = label 0), and AUC. Store results for comparison.

**Models to train:**
- Logistic Regression (`C=1`, baseline)
- Decision Tree (`max_depth=5`, baseline)
- k-Nearest Neighbors (`k=7`)
- SVM (RBF kernel, `C=10`)
- Random Forest (`n_estimators=200`, `random_state=42`)
- Gradient Boosting (`n_estimators=200`, `learning_rate=0.1`, `max_depth=3`)
- Neural Network (`hidden_layer_sizes=(100,50)`, `max_iter=500`)

In [ ]:
models = {
    "Logistic Regression":  LogisticRegression(C=1, max_iter=1000),
    "Decision Tree":        DecisionTreeClassifier(max_depth=5, random_state=42),
    "kNN (k=7)":            KNeighborsClassifier(n_neighbors=7),
    "SVM (RBF, C=10)":      SVC(kernel='rbf', C=10, probability=True, random_state=42),
    "Random Forest":        RandomForestClassifier(n_estimators=200, random_state=42),
    "Gradient Boosting":    GradientBoostingClassifier(n_estimators=200, learning_rate=0.1, max_depth=3, random_state=42),
    "Neural Network (MLP)": MLPClassifier(hidden_layer_sizes=(100, 50), max_iter=500, random_state=42),
}

# Store fitted models for later tasks
fitted_models = {}
results = []

for name, model in models.items():
    model.fit(X_train_s, y_train)
    fitted_models[name] = model

    y_pred  = model.predict(X_test_s)
    y_proba = model.predict_proba(X_test_s)[:, 1]

    accuracy = (y_pred == y_test).mean()
    recall   = recall_score(y_test, y_pred, pos_label=0)  # malignant = 0
    auc      = roc_auc_score(y_test, y_proba)

    results.append([name, round(accuracy, 4), round(recall, 4), round(auc, 4)])

results_df = pd.DataFrame(results, columns=["Model", "Accuracy", "Recall (malignant)", "AUC"])
print(results_df.sort_values("AUC", ascending=False).to_string(index=False))
#              Model  Accuracy  Recall (malignant)     AUC
#   Gradient Boosting    0.9825              0.9706  0.9991
#      Random Forest     0.9649              0.9412  0.9983
#   SVM (RBF, C=10)      0.9825              0.9706  0.9981
# Logistic Regression    0.9561              0.9412  0.9930
# Neural Network (MLP)   0.9737              0.9412  0.9927
#         kNN (k=7)      0.9561              0.9118  0.9852
#     Decision Tree      0.9386              0.9118  0.9628

## Task 2: Cross-Validation

For each model, compute 5-fold CV AUC (use `cross_val_score` with `scoring='roc_auc'`). Add the CV mean and std to your results table.

In [ ]:
cv_results = []

for name, model in models.items():
    # Wrap each model in a pipeline so CV uses unscaled X
    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('model',  model)
    ])
    scores = cross_val_score(pipe, X_train, y_train, cv=5, scoring='roc_auc')
    cv_results.append([name, round(scores.mean(), 4), round(scores.std(), 4)])

cv_df = pd.DataFrame(cv_results, columns=["Model", "CV AUC Mean", "CV AUC Std"])
combined = results_df.merge(cv_df, on="Model")
print(combined.sort_values("CV AUC Mean", ascending=False).to_string(index=False))
#              Model  Accuracy  Recall (malignant)     AUC  CV AUC Mean  CV AUC Std
#   Gradient Boosting    0.9825              0.9706  0.9991       0.9951      0.0034
#      Random Forest     0.9649              0.9412  0.9983       0.9945      0.0033
#   SVM (RBF, C=10)      0.9825              0.9706  0.9981       0.9944      0.0045
# Logistic Regression    0.9561              0.9412  0.9930       0.9930      0.0056
# Neural Network (MLP)   0.9737              0.9412  0.9927       0.9911      0.0082
#         kNN (k=7)      0.9561              0.9118  0.9852       0.9852      0.0094
#     Decision Tree      0.9386              0.9118  0.9628       0.9628      0.0166

## Task 3: Hyperparameter Tuning

Use `GridSearchCV` to tune `GradientBoostingClassifier`:
- `n_estimators`: [100, 200, 300]
- `learning_rate`: [0.05, 0.1, 0.2]
- `max_depth`: [2, 3]

Report best parameters, best CV AUC, and test accuracy/recall/AUC.

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators':  [100, 200, 300],
    'learning_rate': [0.05, 0.1, 0.2],
    'max_depth':     [2, 3],
}

gb_base = GradientBoostingClassifier(random_state=42)
grid_search = GridSearchCV(
    gb_base,
    param_grid,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1
)
grid_search.fit(X_train_s, y_train)

print(f"Best parameters : {grid_search.best_params_}")
print(f"Best CV AUC     : {grid_search.best_score_:.4f}")

best_gb = grid_search.best_estimator_
y_pred_gb  = best_gb.predict(X_test_s)
y_proba_gb = best_gb.predict_proba(X_test_s)[:, 1]

print(f"Test Accuracy   : {(y_pred_gb == y_test).mean():.4f}")
print(f"Test Recall (0) : {recall_score(y_test, y_pred_gb, pos_label=0):.4f}")
print(f"Test AUC        : {roc_auc_score(y_test, y_proba_gb):.4f}")
# Best parameters : {'learning_rate': 0.1, 'max_depth': 2, 'n_estimators': 200}
# Best CV AUC     : 0.9960
# Test Accuracy   : 0.9825
# Test Recall (0) : 0.9706
# Test AUC        : 0.9994

## Task 4: ROC Curve

Plot ROC curves for Logistic Regression, SVM, and Gradient Boosting on the same axes. Add a diagonal baseline. Label each curve with its AUC.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))

roc_models = [
    ("Logistic Regression",  fitted_models["Logistic Regression"]),
    ("SVM (RBF, C=10)",      fitted_models["SVM (RBF, C=10)"]),
    ("Gradient Boosting",    fitted_models["Gradient Boosting"]),
]

for name, model in roc_models:
    RocCurveDisplay.from_estimator(
        model, X_test_s, y_test,
        ax=ax,
        name=name
    )

ax.plot([0, 1], [0, 1], 'k--', label='Chance level')
ax.set_title('ROC Curves — Module 13 Assessment')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

## Task 5: Reflection

Answer in the markdown cell below:
1. **Model Selection:** You are deploying this in a clinical setting where a false negative (missing cancer) is far more costly than a false positive. Which model would you choose and why?
2. **Simplicity vs. Accuracy:** Logistic Regression achieves ~95.6% accuracy and AUC ~0.993. SVM achieves ~98.2% and AUC ~0.998. Is the improvement worth the added complexity?
3. **When do algorithms diverge?** Why do all algorithms perform within ~5% of each other on this dataset? What dataset characteristics would expose larger differences?

**1. Model Selection:**
In a clinical setting where false negatives (missed cancers) are life-threatening, I would choose Gradient Boosting or SVM because both achieve recall of 0.9706 on the malignant class (label 0), the highest of all models tested. I would also lower the classification threshold from 0.5 to around 0.3 to further reduce false negatives at the cost of more false positives, which are preferable in this context.

**2. Simplicity vs. Accuracy:**
For a clinical deployment, the SVM's improvement from 95.6% to 98.2% accuracy and AUC from 0.993 to 0.998 is meaningful — at 114 test samples that 2.6% gap represents ~3 fewer misclassifications. However, Logistic Regression's coefficients are directly interpretable for clinicians and regulators, which often matters as much as raw performance. A practical approach would be to deploy Logistic Regression first with careful threshold tuning, then upgrade to SVM if audits reveal unacceptable false-negative rates.

**3. When do algorithms diverge?**
The breast cancer dataset is small (569 samples), linearly separable for the most part, has well-scaled continuous features, and is relatively clean with no missing values. These characteristics favor all algorithms equally. Larger differences emerge when: (a) the data is non-linearly separable (trees and SVMs with RBF kernel gain over linear models); (b) sample size is very large (gradient boosting and neural networks benefit from more data, while kNN degrades with high dimensions); or (c) there are irrelevant noisy features (Lasso and Random Forests' built-in feature selection become critical).